In [1]:
!pip install fastapi uvicorn nest-asyncio pyngrok
!pip install faiss-cpu sentence-transformers python-jose sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 51.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.8/150.8 kB 7.0 MB/s eta 0:00:00


In [2]:
import nest_asyncio
nest_asyncio.apply()

In [3]:
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker

engine = create_engine("sqlite:///./test.db")
SessionLocal = sessionmaker(bind=engine)
Base = declarative_base()

class User(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True)
    username = Column(String)
    password = Column(String)
    role = Column(String)

class Document(Base):
    __tablename__ = "documents"
    id = Column(Integer, primary_key=True)
    title = Column(String)
    company_name = Column(String)
    document_type = Column(String)
    content = Column(String)

Base.metadata.create_all(bind=engine)

/tmp/ipykernel_9693/3061512041.py:7: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


In [4]:
from jose import jwt
from datetime import datetime, timedelta

SECRET_KEY = "secret123"

def create_token(user_id):
    payload = {
        "user_id": user_id,
        "exp": datetime.utcnow() + timedelta(hours=2)
    }
    return jwt.encode(payload, SECRET_KEY, algorithm="HS256")

In [5]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

index = faiss.IndexFlatL2(384)
chunks_store = []

def chunk_text(text):
    return text.split(".")

def add_document_rag(text):
    chunks = chunk_text(text)
    for chunk in chunks:
        emb = model.encode(chunk)
        index.add(np.array([emb]).astype("float32"))
        chunks_store.append(chunk)

def search_rag(query):
    emb = model.encode(query)
    D, I = index.search(np.array([emb]).astype("float32"), 5)
    return [chunks_store[i] for i in I[0]]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
from fastapi import FastAPI

app = FastAPI()

# ---------- AUTH ----------

@app.post("/auth/register")
def register(username: str, password: str, role: str):
    db = SessionLocal()
    user = User(username=username, password=password, role=role)
    db.add(user)
    db.commit()
    return {"msg": "User registered"}

@app.post("/auth/login")
def login(username: str, password: str):
    db = SessionLocal()
    user = db.query(User).filter(User.username==username).first()
    if user and user.password == password:
        token = create_token(user.id)
        return {"token": token}
    return {"error": "Invalid credentials"}

# ---------- DOCUMENT APIs ----------

@app.post("/documents/upload")
def upload(title: str, company: str, doc_type: str, content: str):
    db = SessionLocal()
    doc = Document(title=title, company_name=company, document_type=doc_type, content=content)
    db.add(doc)
    db.commit()

    add_document_rag(content)

    return {"msg": "Document stored"}

@app.get("/documents")
def get_all():
    db = SessionLocal()
    docs = db.query(Document).all()
    return docs

@app.get("/documents/{doc_id}")
def get_doc(doc_id: int):
    db = SessionLocal()
    doc = db.query(Document).filter(Document.id==doc_id).first()
    return doc

@app.delete("/documents/{doc_id}")
def delete_doc(doc_id: int):
    db = SessionLocal()
    doc = db.query(Document).filter(Document.id==doc_id).first()
    db.delete(doc)
    db.commit()
    return {"msg": "Deleted"}

@app.get("/documents/search")
def search_metadata(company: str):
    db = SessionLocal()
    docs = db.query(Document).filter(Document.company_name==company).all()
    return docs

# ---------- RAG ----------

@app.post("/rag/index-document")
def index_doc(text: str):
    add_document_rag(text)
    return {"msg": "Indexed"}

@app.post("/rag/search")
def rag_search(query: str):
    return {"results": search_rag(query)}

@app.get("/rag/context/{doc_id}")
def get_context(doc_id: int):
    db = SessionLocal()
    doc = db.query(Document).filter(Document.id==doc_id).first()
    return {"content": doc.content}

In [8]:
import uvicorn

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()

INFO:     Started server process [9693]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     2409:40e3:40ed:8a3b:9001:2bf3:161:44c8:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     2409:40e3:40ed:8a3b:9001:2bf3:161:44c8:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     2409:40e3:40ed:8a3b:9001:2bf3:161:44c8:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     2409:40e3:40ed:8a3b:9001:2bf3:161:44c8:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     2409:40e3:40ed:8a3b:9001:2bf3:161:44c8:0 - "POST /auth/register?username=Prateek%20&password=123&role=admin HTTP/1.1" 200 OK
INFO:     2409:40e3:40ed:8a3b:9001:2bf3:161:44c8:0 - "POST /auth/register?username=Prateek%20&password=123&role=admin HTTP/1.1" 200 OK
INFO:     2409:40e3:40ed:8a3b:9001:2bf3:161:44c8:0 - "POST /auth/login?username=Prateek&password=123 HTTP/1.1" 200 OK
INFO:     2409:40e3:40ed:8a3b:9001:2bf3:161:44c8:0 - "POST /auth/login?username=Prateek&password=123 HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [9693]


In [7]:
from pyngrok import ngrok

# 🔑 Your actual token (already added)
ngrok.set_auth_token("3COQY4YESSAK43EVL61hiM8E1fU_3xPjEmcAqPLXS8jzaWcgz")

# 🔗 Create public tunnel
public_url = ngrok.connect(addr=8000, bind_tls=True)

print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://headcount-douche-isolation.ngrok-free.dev" -> "http://localhost:8000"


In [10]:
!git clone https://github.com/Psty12/FastAPI-Financial-Document-Management-with-Semantic-Analysis.git

Cloning into 'FastAPI-Financial-Document-Management-with-Semantic-Analysis'...


In [11]:
!ls

FastAPI-Financial-Document-Management-with-Semantic-Analysis  test.db
sample_data


In [14]:
!cp /content/project.ipynb /content/FastAPI-Financial-Document-Management-with-Semantic-Analysis/

cp: cannot stat '/content/project.ipynb': No such file or directory


In [15]:
!ls /content


FastAPI-Financial-Document-Management-with-Semantic-Analysis  test.db
sample_data
